In [1]:
#imports
import numpy as np
import pandas as pd
import csv
import glob
import traceback
import sys
from IPython.display import clear_output, display

In [4]:
#global variables
alph='abcdefghijklmnopqrstuvwxyz'
alph_list=[i for i in alph]
filename='subtlex.csv'
weight='FREQcount'
n='\n'

In [5]:
#file loading
corpus=pd.read_csv(filename,dtype={'Word': str}).set_index('Word')
#freq_df=pd.DataFrame(columns=alph_list+['Word','Index']).set_index(['Word','Index'])
freq_df=pd.read_csv('freq_df.csv').set_index(['Word','Index'])

In [6]:
#file saving
freq_df.to_csv("freq_df.csv", index=True)

In [7]:
#testing
freq_df.size

2368938

In [8]:
#functions
#print(corpus)
def get_dict(expr,weight=weight,corpus=corpus,freq_df=freq_df):
    '''
    expr is a regex expression as a string
    '''
    n=len(expr)
    ret=[{char:0 for char in alph} for i in range(n)]
    corpus_df=corpus.filter(regex='^'+expr+'$',axis=0)
    for word,freq in zip(corpus_df.index,corpus_df[weight]):
        for i in range(n):
            ret[i][str(word).lower()[i]]+=freq
    
    for i in range(n):
        temp_dict=ret[i]
        total=sum([temp_dict[key] for key in alph_list])
        if total:
            freq_df.loc[(expr,i),:]={key:temp_dict[key]/total for key in alph_list}
        else:
            raise SearchFailure
    #print(ret)
    return ret

In [10]:
#classes
class crossword:
    def __init__(self,arr,corpus=corpus,weight=weight,freq_df=freq_df,display_freq=1):
        '''
        arr is array with shape (h,w); entries are either letters or a period or asterisk
        '''
        if type(arr)==str:
            arr=[[char for char in row] for row in arr.split('\n')]
            #print(arr)
        self.arr=np.array(arr)
        self.h,self.w=self.arr.shape
        
        self.corpus=corpus
        self.weight=weight
        self.freq_df=freq_df
        self.order=[]
        self.iteration=0
        self.display_freq=display_freq
    
    def get_dict(self,cell):
        i,acr=self.get_word(cell,'across')
        j,down=self.get_word(cell,'down') 
        try:
            acr=self.freq_df.loc[(acr,i)]
        except:
            acr=get_dict(acr)[i]
        try:
            down=self.freq_df.loc[(down,j)]
        except:
            down=get_dict(down)[j]
        ret={key:acr[key]*down[key] for key in alph_list}
        total=sum([ret[key] for key in alph_list])
        if total:
            return {key:ret[key]/total for key in alph_list}
        else:
            raise SearchFailure(f'no valid characters at {cell} (self.get_dict)')

    def get_next_cell(self):
        '''
        Adds next best cell to consider to order list
        Returns None
        '''
        arr=np.zeros((self.h,self.w))
        for y in range(self.h):
            for x in range(self.w):
                if self.arr[y][x]=='.':
                    try:
                        arr[y][x]=self.get_max_probability((x,y))
                    except SearchFailure as e:
                        if 'block'==str(e):
                            print('somehow this error got raised; it should not be an issue but this should not have happened')
                            arr[y][x]=0
                        else:
                            raise
                    except:
                        raise
                else:
                    arr[y][x]=0
        
        y,x=np.unravel_index(np.argmax(arr),arr.shape)
        self.order.append((x,y))
                
        
    def get_max_probability(self,cell):
        try:
            freq=self.get_dict(cell)
            return max(freq.values())
        except SearchFailure:
            raise SearchFailure(f'no valid characters at {cell}')
        
    def recursive_fill(self,n=0):
        self.iteration+=1
        if self.display_freq:
            if self.iteration%self.display_freq==0:
                clear_output(wait=True)
                print(f'n={n},iteration={self.iteration}')
                print(self)
        try: 
            if np.count_nonzero(self.arr=='.'):
                if n>=len(self.order): 
                    self.get_next_cell() #can raise error
                cell=self.order[n]
                x,y=cell
                #print(cell)
                #print(type(x))
                freq=self.get_dict(cell) #can raise errors
                chars=[key for key in freq if freq[key]]
                weights=[freq[key] for key in freq if freq[key]]
            
                for char in np.random.choice(chars, size=len(chars),replace=False,p=weights):
                    try:
                        
                        self.arr[y][x]=char
                        #print(self)
                        self.recursive_fill(n+1)
                        return True 
                    except RecursionError:
                        raise
                    except:
                        pass
                self.arr[y][x]='.'
                #print('failed')
                raise SearchFailure(f'no valid characters found in cell {n}, located at {self.order[n]}') #if all fail
            else:
                #print(n)
                return True
        except Exception as e:
            #print(e)
            tbe = traceback.TracebackException.from_exception(e, capture_locals=True)
            # Prepend the outer stack frames leading up to the try block
            outer_frames = traceback.extract_stack()[:-1]
            tbe.stack = traceback.StackSummary.from_list(outer_frames + list(tbe.stack))
            #print("".join(tbe.format()))
            raise
        except:
            raise
            
            
        
    def get_word(self,cell,direction):
        '''
        cell=(x,y) both integers
        direction='across' or 'down'
        '''
        x,y=cell
        if self.arr[y][x]=='*':
            raise SearchFailure('block')
        ret=self.arr[y][x]
        i=0
        if direction=='across':
            x_=x
            while x_>0:
                x_-=1
                if self.arr[y][x_]=='*':
                    break
                ret=self.arr[y][x_]+ret
                i+=1
            x_=x
            while x_<self.w-1:
                x_+=1
                if self.arr[y][x_]=='*':
                    break
                ret+=self.arr[y][x_]
        elif direction=='down':
            y_=y
            while y_>0:
                y_-=1
                if self.arr[y_][x]=='*':
                    break
                ret=self.arr[y_][x]+ret
                i+=1
            y_=y
            while y_<self.h-1:
                y_+=1
                if self.arr[y_][x]=='*':
                    break
                ret+=self.arr[y_][x]
        else:
            print('invalid direction')
            raise ValueError('invalid direction')
        
        return i,ret
        
    def __str__(self):
        return '\n'.join([' '.join(row) for row in self.arr])
        #return f'  {" ".join([str(i) for i in range(self.w)])}\n{n.join([f"{i} {row}" for i,row in enumerate(" ".join(row) for row in self.arr)])}'

class SearchFailure(Exception):
    pass

test=crossword('''...
...
...''')
print(test)

. . .
. . .
. . .


In [11]:
#print(freq_df.loc[('...p.....',1)])

expr='....n..g........'
#print(freq_df.loc['....n..g........'])
get_dict(expr)

SearchFailure: 

In [96]:
thurs_str='''
...p.....**...*disact
...o.....**...*..a...
...i.....*i...*..i...
***n...*recnaissance*
*ind*.....i..*...e.**
*..t.*....c....*.d...
w..e..*...**.....*...
iegrate**........*...
e..*..n..*....*......
r....*cracter*....***
*...*.n.........*...*
***...t*.......*.....
......*....*.....*...
...*........**.......
...*.....**...*......
.....*.........*..***
**untwt*........*...*
*....t.......*......*
.....f*....*.........
.....u*...**.........
.....l*...**abilities
'''[1:-1]
thurs=crossword(thurs_str,display_freq=1)
thurs.recursive_fill()

n=1,iteration=2
  0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20
0 . . . p . . . . . * * . . . * d i s a c t
1 . . . o . . . . . * * . . . * . . a . . .
2 . . . i . . . . . * i . f . * . . i . . .
3 * * * n . . . * r e c n a i s s a n c e *
4 * i n d * . . . . . i . . * . . . e . * *
5 * . . t . * . . . . c . . . . * . d . . .
6 w . . e . . * . . . * * . . . . . * . . .
7 i e g r a t e * * . . . . . . . . * . . .
8 e . . * . . n . . * . . . . * . . . . . .
9 r . . . . * c r a c t e r * . . . . * * *
10 * . . . * . n . . . . . . . . . * . . . *
11 * * * . . . t * . . . . . . . * . . . . .
12 . . . . . . * . . . . * . . . . . * . . .
13 . . . * . . . . . . . . * * . . . . . . .
14 . . . * . . . . . * * . . . * . . . . . .
15 . . . . . * . . . . . . . . . * . . * * *
16 * * u n t w t * . . . . . . . . * . . . *
17 * . . . . t . . . . . . . * . . . . . . *
18 . . . . . f * . . . . * . . . . . . . . .
19 . . . . . u * . . . * * . . . . . . . . .
20 . . . . . l * . . . * * a b i l i t 

SearchFailure: no valid characters found in cell 0, located at (np.int64(12), np.int64(2))

In [ ]:
monday_str='''
.....*....*...*
.....*........*
..............*
...*....**.....
**....**.......
...........*...
....*.......***
.....*...*.....
***.......*....
...*...........
.......**....**
.....**....*...
*..............
*........*.....
*...*....*.....
'''[1:-1]
monday=crossword(monday_str,display_freq=1000)
monday.recursive_fill()
freq_df.to_csv("freq_df.csv", index=True)

print(monday)

n=149,iteration=1366000
t . i l s * s . . . * . . . *
. . m a c * h . . . g . a m *
a l p h a b e t i c a l l y *
. . l * t r e e * * l . . . .
* * a . . o * * s . a . . . .
c o n t e n t m e n t * . . .
. . t . * c h a r g e s * * *
a . s . p * o n e * a t t e r
* * * l a t r i n e * o o s e
l b w * c o n c e n t r a t e
a r i d i t y * * t r e s * *
g o n i f * * u s e e * t w o
* a d m i n i s t r a t i o n
* d u l c i n e a * t h e r e
* s p y * . . . . * y e s e s


In [10]:
print(monday)

r e c a p * j o s e * s c h *
s t a l l * o k o l e h a o *
u n p l e a s a n t n e s s *
m a t * a c h y * * v i s t a
* * u r d u * * e z e k i e l
p a r a s t e r n a l * a l l
a d e n * e y e d r o p * * *
v o d k a * e s e * p i c k s
* * * s u f f e r s * l o o t
b a g * d e u t s c h l a n d
e z e k i e l * * o a s t * *
d o n a t * * c r t s * i s h
* r e c o n n a i s s a n c e
* e v e r y o n e * e d g a r
* s a y * e w e s * l e s t e


In [88]:
lis='''
.e.*...
.z..*..
.e.....
*kacey*
.i.....
.e*....
.l.*...
'''[1:-1]




kacey_7x7=crossword(lis,display_freq=100)
kacey_7x7.recursive_fill()
print(kacey_7x7)

n=12,iteration=100
. e . * . . .
e z r a * . .
l e e c h e s
* k a c e y *
. i . r i . .
. e * a . . .
. l . * . . .
g e t * t c h
e z r a * h i
l e e c h e s
* k a c e y *
c i t r i n e
h e * a r e a
a l l * s y m


In [67]:
lis='''
ccher****
a.....***
l......**
o..*....*
g...**...
*...*csup
**...t...
***..l...
****.e...
'''[1:-1]

cat_9x9=crossword(lis,display_freq=100)
cat_9x9.recursive_fill()
print(cat_9x9)


n=35,iteration=185100
c c h e r * * * *
a l i y a h * * *
l a t e r a l * *
o w t * a b a . *
g . e . * * s o u
* . r . * c s u p
* * s t a t o r s
* * * a l l e n e
* * * * . e s e t
c c h e r * * * *
a l i y a h * * *
l a t e r a l * *
o r t * e d a m *
g e e k * * s o u
* t r i * c s u p
* * s t a t o r s
* * * s e l e n e
* * * * r e s e t


In [58]:
'''*ccm
roue
entr
ade*'''

'''*boh
roue
entr
ads*'''

'''*ary
knee
lags
oar*'''

'''*wry
chee
herm
itd*'''

'''*rcd
roue
emer
gps*'''

'''*you
noun
meta
alp*'''

'*you\nnoun\nmeta\nalp*'

In [ ]:
'''
a f b * s l t
h o e * c a i
a r c s i n e
* k a c e y *
d i u r n a l
i n s * c r e
a g e * e d s
'''


'''
r e c a p * j o s e * s c h *
s t a l l * o k o l e h a o *
u n p l e a s a n t n e s s *
m a t * a c h y * * v i s t a
* * u r d u * * e z e k i e l
p a r a s t e r n a l * a l l
a d e n * e y e d r o p * * *
v o d k a * e s e * p i c k s
* * * s u f f e r s * l o o t
b a g * d e u t s c h l a n d
e z e k i e l * * o a s t * *
d o n a t * * c r t s * i s h
* r e c o n n a i s s a n c e
* e v e r y o n e * e d g a r
* s a y * e w e s * l e s t e
'''

'''
  0 1 2 3 4 5 6 7 8 9 10 11 12 13 14
0 c o s e t * w a g s * m s s *
1 a b u s e * i r r i t a t e *
2 w o r c e s t e r s h i r e *
3 s e g * t h e n * * e d e m a
4 * * i d e e * * t e r s e l y
5 d e c e r e b r a t e * t y e
6 i r a n * t e a s h o p * * *
7 v a l i d * a t t * f e s s e
8 * * * m e a t i e r * a w o l
9 a f t * f l u o r e s c e n t
10 p r o p e l s * * a w e e * *
11 r i p e n * * a b l e * t w o
12 * s p e c i a l i s a t i o n
13 * c e r e b r a l * t h e r e
14 * o r s * m e r l * y e s e s

'''



'''
r e c a p * j o s e * s c h *
s t a l l * o k o l e h a o *
u n p l e a s a n t n e s s *
m a t * a c h y * * v i s t a
* * u r d u * * e z e k i e l
p a r a s t e r n a l * a l l
a d e n * e y e d r o p * * *
v o d k a * e s e * p i c k s
* * * s u f f e r s * l o o t
b a g * d e u t s c h l a n d
e z e k i e l * * o a s t * *
d o n a t * * c r t s * i s h
* r e c o n n a i s s a n c e
* e v e r y o n e * e d g a r
* s a y * e w e s * l e s t e
'''